# PLUTO: from vector scene to trajectory

This offline lesson implements a **simplified educational variant**, with randomly initialized weights and toy synthetic data. It does not reproduce nuPlan or the official code. See `docs/paper-to-code.md` in the repository.




In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(7)
torch.set_num_threads(1)


## Data contract

Masks use `True` for valid points. The first agent is ego; positions are metres in the ego frame. This synthetic fixture supplies mode labels directly; an elementary projection helper is available but is not connected to real preprocessing. Shapes below use batch size 2 and a four-step horizon.




In [ ]:
"""Deterministic synthetic input fixture, not a nuPlan adapter."""


def sample_batch(batch_size=2, future_steps=4, seed=42):
    g = torch.Generator().manual_seed(seed)
    rand = lambda *shape: torch.randn(*shape, generator=g)
    b, a, r = batch_size, 3, 2
    target = rand(b, future_steps, 6) * 0.1
    return {'agents': rand(b, a, 5, 9), 'agent_mask': torch.ones(b, a, 5, dtype=torch.bool),
            'ego_state': rand(b, 6), 'map': rand(b, 3, 5, 6), 'map_mask': torch.ones(b, 3, 5, dtype=torch.bool),
            'static': rand(b, 2, 5), 'static_mask': torch.ones(b, 2, dtype=torch.bool),
            'reference': rand(b, r, 6, 6), 'reference_mask': torch.ones(b, r, 6, dtype=torch.bool),
            'target': target, 'target_mask': torch.ones(b, future_steps, dtype=torch.bool),
            'agent_target': rand(b, a - 1, future_steps, 6) * 0.1,
            'agent_target_mask': torch.ones(b, a - 1, future_steps, dtype=torch.bool),
            'mode_label': torch.zeros(b, dtype=torch.long)}

batch = sample_batch(batch_size=2, future_steps=4)
print({k: tuple(v.shape) for k, v in batch.items()})


## Architecture

Pointwise embeddings are pooled over valid elements. Queries combine one reference-line token and one learned longitudinal mode. Decoder layers attend across references, then modes, then encoded scene tokens. Empty references use the free head.




In [ ]:
"""Vector scene encoder and factorized lateral/longitudinal planner.

All spatial inputs are in the ego frame, in metres. Masks are True for valid
entries. The first agent is ego; other agent future predictions are agent-relative.
"""
from dataclasses import dataclass
import torch
from torch import nn


@dataclass
class PlutoConfig:
    dim: int = 128
    heads: int = 4
    encoder_layers: int = 4
    decoder_layers: int = 4
    modes: int = 12
    future_steps: int = 80
    dropout: float = 0.1


def mlp(n_in, hidden, n_out):
    return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, n_out))


class PointEncoder(nn.Module):
    """Masked pointwise MLP and max pooling; empty polylines map to zero."""
    def __init__(self, channels, dim):
        super().__init__()
        self.net = mlp(channels, dim, dim)

    def forward(self, x, mask):
        y = self.net(x).masked_fill(~mask.unsqueeze(-1), float('-inf'))
        pooled = y.max(dim=-2).values
        return torch.where(mask.any(dim=-1, keepdim=True), pooled, torch.zeros_like(pooled))


class DecoderLayer(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.lateral = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.longitudinal = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.cross = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(4)])
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, q, memory, ref_valid, scene_valid):
        b, r, m, d = q.shape
        x = q.transpose(1, 2).reshape(b * m, r, d)
        # An all-masked row yields NaNs in attention; a temporary dummy is ignored later.
        valid = ref_valid.clone()
        valid[~valid.any(-1), 0] = True
        mask = (~valid)[:, None, :].expand(b, m, r).reshape(b * m, r)
        y = self.norms[0](x)
        x = x + self.drop(self.lateral(y, y, y, key_padding_mask=mask, need_weights=False)[0])
        x = x.reshape(b, m, r, d).transpose(1, 2).reshape(b * r, m, d)
        y = self.norms[1](x)
        x = x + self.drop(self.longitudinal(y, y, y, need_weights=False)[0])
        x = x.reshape(b, r * m, d)
        y = self.norms[2](x)
        x = x + self.drop(self.cross(y, memory, memory, key_padding_mask=~scene_valid, need_weights=False)[0])
        x = x + self.drop(self.ff(self.norms[3](x)))
        return x.reshape(b, r, m, d)


class Pluto(nn.Module):
    """PLUTO-style model. See docs/paper-to-code.md for deliberate differences."""
    def __init__(self, config=PlutoConfig()):
        super().__init__()
        self.config = config
        d = config.dim
        self.agent_encoder = PointEncoder(9, d)
        self.map_encoder = PointEncoder(6, d)
        self.static_encoder = mlp(5, d, d)
        self.ego_encoder = mlp(6, d, d)
        self.type_embedding = nn.Embedding(3, d)
        layer = nn.TransformerEncoderLayer(d, config.heads, d * 4, config.dropout, batch_first=True, norm_first=True)
        self.scene_encoder = nn.TransformerEncoder(layer, config.encoder_layers, norm=nn.LayerNorm(d))
        self.reference_encoder = PointEncoder(6, d)
        self.mode = nn.Parameter(torch.randn(config.modes, d) * 0.01)
        self.query_projection = nn.Linear(2 * d, d)
        self.decoder = nn.ModuleList([DecoderLayer(d, config.heads, config.dropout) for _ in range(config.decoder_layers)])
        self.trajectory_head = mlp(d, 2 * d, config.future_steps * 6)
        self.score_head = mlp(d, d, 1)
        self.prediction_head = mlp(d, 2 * d, config.future_steps * 6)
        self.free_head = mlp(d, 2 * d, config.future_steps * 6)
        self.projection = mlp(d, d, d)

    def forward(self, batch):
        c = self.config
        agents, maps, static = batch['agents'], batch['map'], batch['static']
        b, a = agents.shape[:2]
        agent_valid = batch['agent_mask'].any(-1)
        # The ego state supplies a valid scene token even if its history is absent.
        agent_valid = agent_valid.clone()
        agent_valid[:, 0] = True
        agent = self.agent_encoder(agents, batch['agent_mask'])
        agent = torch.cat([self.ego_encoder(batch['ego_state']).unsqueeze(1), agent[:, 1:]], dim=1)
        tokens = torch.cat([agent + self.type_embedding.weight[0],
                            self.map_encoder(maps, batch['map_mask']) + self.type_embedding.weight[1],
                            self.static_encoder(static) + self.type_embedding.weight[2]], dim=1)
        scene_valid = torch.cat([agent_valid, batch['map_mask'].any(-1), batch['static_mask']], dim=1)
        memory = self.scene_encoder(tokens, src_key_padding_mask=~scene_valid)
        prediction = self.prediction_head(memory[:, 1:a]).reshape(b, a - 1, c.future_steps, 6)
        free = self.free_head(memory[:, 0]).reshape(b, c.future_steps, 6)
        refs, ref_mask = batch['reference'], batch['reference_mask']
        r = refs.shape[1]
        if r == 0:
            return {'trajectory': None, 'logits': None, 'prediction': prediction, 'free': free,
                    'selected': free, 'hidden': self.projection(memory[:, 0])}
        ref_valid = ref_mask.any(-1)
        ref = self.reference_encoder(refs, ref_mask)
        q = self.query_projection(torch.cat([ref.unsqueeze(2).expand(b, r, c.modes, c.dim),
                                             self.mode[None, None].expand(b, r, c.modes, c.dim)], -1))
        for layer in self.decoder:
            q = layer(q, memory, ref_valid, scene_valid)
        trajectory = self.trajectory_head(q).reshape(b, r, c.modes, c.future_steps, 6)
        logits = self.score_head(q).squeeze(-1).masked_fill(~ref_valid.unsqueeze(-1), -1e9)
        best = logits.reshape(b, -1).argmax(-1)
        selected = trajectory.reshape(b, r * c.modes, c.future_steps, 6)[torch.arange(b, device=best.device), best]
        selected = torch.where(ref_valid.any(-1)[:, None, None], selected, free)
        return {'trajectory': trajectory, 'logits': logits, 'prediction': prediction, 'free': free,
                'selected': selected, 'hidden': self.projection(memory[:, 0])}


## Assemble and inspect

The toy network uses D=16 and one layer per stack, unlike the paper configuration (D=128, four layers and 12 longitudinal queries).




In [ ]:
cfg = PlutoConfig(dim=16, heads=4, encoder_layers=1, decoder_layers=1, modes=3, future_steps=4, dropout=0.0)
model = Pluto(cfg)
out = model(batch)
print('plans', out['trajectory'].shape, 'other agents', out['prediction'].shape)


## Objectives and optimization

The toy imitation loss uses a supplied reference/mode label, Smooth L1 trajectory and prediction terms and classification. The paper also trains with collision/off-road and contrastive terms; the standalone utilities below are not wired into this toy training step. SDF lookup is differentiable bilinear interpolation.




In [ ]:
"""Planning, agent prediction, contrastive and differentiable SDF objectives."""
import torch
import torch.nn.functional as F


def masked_smooth_l1(pred, target, valid):
    errors = F.smooth_l1_loss(pred, target, reduction='none').sum(-1)
    return (errors * valid.to(errors.dtype)).sum() / valid.sum().clamp(min=1)


def contrastive_loss(anchor, positive, negative, temperature=0.1, valid=None):
    if temperature <= 0:
        raise ValueError('temperature must be positive')
    a, p, n = (F.normalize(x, dim=-1) for x in (anchor, positive, negative))
    logits = torch.stack([(a * p).sum(-1), (a * n).sum(-1)], -1) / temperature
    loss = F.cross_entropy(logits, torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device), reduction='none')
    if valid is None:
        return loss.mean() if loss.numel() else logits.sum() * 0
    return (loss * valid).sum() / valid.sum().clamp(min=1)


def sdf_loss(trajectory, sdf, resolution=0.2, offsets=(-1., 0., 1.), radius=1., margin=0.):
    """Eq. 12: average sum of hinge penalties per timestep; x right, y up.

    SDF is positive in safe space, shape [B,H,W], with its origin at the
    center pixel. Out-of-bounds positions get zero distance (conservative).
    """
    if resolution <= 0:
        raise ValueError('resolution must be positive')
    h, w = sdf.shape[-2:]
    if h < 2 or w < 2:
        raise ValueError('SDF dimensions must be at least 2')
    off = trajectory.new_tensor(offsets).view(1, 1, -1, 1)
    centers = trajectory[..., None, :2] + off * trajectory[..., None, 2:4]
    gx = 2 * (centers[..., 0] / resolution + (w - 1) / 2) / (w - 1) - 1
    gy = 2 * (-centers[..., 1] / resolution + (h - 1) / 2) / (h - 1) - 1
    grid = torch.stack((gx, gy), -1)
    distance = F.grid_sample(sdf.unsqueeze(1), grid, mode='bilinear', padding_mode='zeros', align_corners=True).squeeze(1)
    return (radius + margin - distance).clamp(min=0).sum(-1).mean()


def imitation_loss(out, batch):
    target, valid = batch['target'], batch['target_mask']
    free = masked_smooth_l1(out['free'], target, valid)
    pred = masked_smooth_l1(out['prediction'], batch['agent_target'], batch['agent_target_mask'])
    if out['trajectory'] is None:
        zero = free * 0
        return {'total': free + pred, 'regression': zero, 'classification': zero, 'free': free, 'prediction': pred}
    traj, logits = out['trajectory'], out['logits']
    b, r, m, t, _ = traj.shape
    ref_valid = batch['reference_mask'].any(-1)
    eligible = ref_valid.any(-1) & valid.any(-1)
    idx = batch['mode_label'].long().clamp(0, r * m - 1)
    selected_valid = ref_valid.gather(1, (idx // m).unsqueeze(1)).squeeze(1)
    if (eligible & ~selected_valid).any():
        raise ValueError('mode_label refers to a masked reference line')
    chosen = traj.reshape(b, r * m, t, 6)[torch.arange(b, device=idx.device), idx]
    reg = masked_smooth_l1(chosen, target, valid & eligible[:, None])
    cls = F.cross_entropy(logits.reshape(b, -1)[eligible], idx[eligible]) if eligible.any() else logits.sum() * 0
    return {'total': reg + cls + free + pred, 'regression': reg, 'classification': cls, 'free': free, 'prediction': pred}


def assign_modes(endpoint, reference_xy, reference_mask, modes):
    """Project [B,2] endpoints onto valid polylines, returning flattened R*M labels.

    Longitudinal bins divide each reference length into M-1 equal intervals;
    the last bin is for projections at or beyond the end. Points are in metres
    in the same coordinate frame. Every sample needs a non-degenerate line.
    """
    if modes < 2 or reference_xy.shape[-2] < 2:
        raise ValueError('need at least two modes and two points per reference')
    start, end = reference_xy[..., :-1, :], reference_xy[..., 1:, :]
    delta = end - start
    length2 = (delta * delta).sum(-1)
    valid = reference_mask[..., :-1] & reference_mask[..., 1:] & (length2 > 1e-12)
    if not valid.any(dim=(-1, -2)).all():
        raise ValueError('every sample needs a valid, non-degenerate reference segment')
    length = length2.sqrt()
    offset = endpoint[:, None, None, :] - start
    u = ((offset * delta).sum(-1) / length2.clamp(min=1e-12)).clamp(0., 1.)
    projected = start + u[..., None] * delta
    distance = ((endpoint[:, None, None, :] - projected) ** 2).sum(-1).masked_fill(~valid, float('inf'))
    b, r, s = distance.shape
    segment_index = distance.reshape(b, -1).argmin(-1)
    ref_index, point_index = segment_index // s, segment_index % s
    lengths = torch.where(valid, length, torch.zeros_like(length))
    accumulated = lengths.cumsum(-1) - lengths
    batch_idx = torch.arange(b, device=endpoint.device)
    arc = accumulated[batch_idx, ref_index, point_index] + u[batch_idx, ref_index, point_index] * length[batch_idx, ref_index, point_index]
    total = lengths.sum(-1)[batch_idx, ref_index]
    longitudinal = torch.floor(arc / (total / (modes - 1)).clamp(min=1e-12)).long().clamp(0, modes - 1)
    return ref_index * modes + longitudinal


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = imitation_loss(out, batch)
optimizer.zero_grad()
losses['total'].backward()
optimizer.step()
print({k: round(v.item(), 4) for k, v in losses.items()})


## Inference and evaluation

Learned logits select a candidate. The paper additionally uses simulation-based scoring, which is absent here. This average displacement error (ADE) is a synthetic open-loop diagnostic, not a nuPlan score.




In [ ]:
model.eval()
with torch.no_grad():
    pred = model(batch)['selected']
    ade = (pred[..., :2] - batch['target'][..., :2]).norm(dim=-1).mean()
print('synthetic ADE (m):', float(ade))


## Checkpoint workflow

The separate `pluto.train` module implements an epoch-boundary synthetic checkpoint containing model, optimizer, scheduler, RNG state and config. Exact resume is only tested for CPU single-process synthetic data. Real data, MLflow and hosted CI remain future work.


